# Figure 6 - Viasat Validation

Regenerates the four validation panels with the same palette, cartographic background and PNG/PDF export convention used in notebooks 01 and 02.


In [63]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm

PALETTE = {
    "src_dest": "#E7D1D1",
    "rnd_rnd": "#DCC7DF",
    "rnd_dest": "#C4DADD",
    "src_rnd": "#C8DEC8",

    "roads_main": "#8D8D8D",
    "roads_minor": "#CFCFCF",

    "text": "#2F2F2F",
    "background": "#FAFAFA",
    "grid": "#DDDDDD",
}

paper_cmap = LinearSegmentedColormap.from_list(
    "paper_activity",
    [
        "#EAF6F7",  # very low
        "#AFCFD4",  # low
        "#D5C0DC",  # medium
        "#D9B8B8",  # high
        "#9E5F63",  # very high
    ],
    N=256,
)

# ---------- GLOBAL MATPLOTLIB STYLE ----------
plt.rcParams.update({

    # Figure
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["background"],
    "savefig.facecolor": PALETTE["background"],

    # Text
    "text.color": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],

    # Font sizes
    "font.size": 18,
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 26,

    # Axes
    "axes.edgecolor": "#BDBDBD",
    "axes.linewidth": 1.2,
    "axes.grid": False,

    # Grid
    "grid.color": PALETTE["grid"],
    "grid.alpha": 0.4,
    "grid.linewidth": 0.8,

    # Legend
    "legend.frameon": True,
    "legend.facecolor": "white",
    "legend.edgecolor": "#DDDDDD",

    # Lines
    "lines.linewidth": 2.2,

    # Ticks
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,

    # DPI
    "figure.dpi": 300,
    "savefig.dpi": 300,

    # PDF/SVG publication quality
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

In [64]:
# Common imports and output utility
# Run from the repository root or from any subdirectory inside the repository.
try:
    import os
    from pathlib import Path
    import warnings

    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import seaborn as sns
    import contextily as ctx

    warnings.filterwarnings("ignore")
    
    year = 2020
    
    start_dir = Path.cwd().resolve()
    ROOT = next(
        (
            candidate
            for candidate in [start_dir, *start_dir.parents]
            if (candidate / "article/tex/epjds_main_draft.tex").exists()
        ),
        None,
    )
    if ROOT is None:
        raise FileNotFoundError("Repository root not found from current working directory")

    os.chdir(ROOT)

    images_dir = Path("article/tex/images")
    images_dir.mkdir(parents=True, exist_ok=True)

    sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)
    plt.rcParams.update({
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    def save_fig_local(fig, stem):
        """Save each panel as both PNG and PDF in the manuscript image folder."""
        stem = Path(stem).stem
        for ext in ("png", "pdf"):
            fig.savefig(images_dir / f"{stem}.{ext}", bbox_inches="tight", dpi=300)
        plt.close(fig)
        print(f"OK: saved {stem}.png and {stem}.pdf in {images_dir}")

    print(f"OK: setup complete; repo root is {ROOT}; image output is {images_dir.resolve()}")

except Exception as exc:
    print(f"ERROR setup: {type(exc).__name__}: {exc}")
    raise


OK: setup complete; repo root is /mnt/c/Users/belli/OneDrive/Desktop/code/covid_hep; image output is /mnt/c/Users/belli/OneDrive/Desktop/code/covid_hep/article/tex/images


In [65]:
# Shared loaders and plotting helpers for Figure 6 validation panels.
try:
    results_dir = Path("article/results")

    CARTO_PATH = Path("article/data/uniq/carto.geojson")
    VIASAT_FLUX_PATH = Path("article/data/viasat_20200210_20200218_timed_fluxes.csv")
    VIASAT_FLUX_PATH_2021 = Path("article/data/viasat_20210628_20210705_timed_fluxes.csv")

    PENETRATION_PATHS = {
        "2020": results_dir / "2020/global/penetrazione/log_eta_20200210_20200217.txt",
        "2021": results_dir / "2021/global/penetrazione/log_eta_20210628_20210704.txt",
    }

    CLUSTER_PATHS = {
        "2020": results_dir / "2020/global/penetrazione/Log/20200210_20200217/Log_clusters.gpkg",
        "2021": results_dir / "2021/global/penetrazione/Log/20210628_20210704/Log_clusters.gpkg",
    }

    # Meaningful labels instead of showing only the raw classes -1, -2, -3.
    # The classes are rounded log10(eta), so they correspond approximately to eta = 10^class.
    PENETRATION_LABELS = {
        -1: r"$\eta \sim 10^{-1}$",
        -2: r"$\eta \sim 10^{-2}$",
        -3: r"$\eta \sim 10^{-3}$",
    }

    PENETRATION_COLORS = {
        -1: PALETTE["roads_main"],
        -2: PALETTE["rnd_dest"],
        -3: PALETTE["src_dest"],
    }

    CLUSTER_COLORS = {
        -1: "#252525",
        -2: "#3182bd",
        -3: "#e6550d",
    }

    def read_roads(path=CARTO_PATH):
        roads = gpd.read_file(path)
        if roads.crs is None:
            roads = roads.set_crs(epsg=4326)
        return roads

    def road_id_column(roads):
        if "poly_cid" in roads.columns:
            return "poly_cid"
        if "cid" in roads.columns:
            return "cid"
        raise KeyError("Neither 'poly_cid' nor 'cid' found in the road network")

    def add_positron_basemap(ax, gdf, zoom="auto"):
        try:
            ctx.add_basemap(
                ax,
                source=ctx.providers.CartoDB.Positron,
                crs=gdf.crs,
                attribution=False,
                zoom=zoom,
                zorder=1,
            )
        except Exception as tile_exc:
            print(f"INFO: basemap unavailable; saving vector-only panel: {tile_exc}")

    def set_map_extent(ax, gdf, margin_fraction=0.035):
        xmin, ymin, xmax, ymax = gdf.total_bounds
        margin = margin_fraction * max(xmax - xmin, ymax - ymin)
        ax.set_xlim(xmin - margin, xmax + margin)
        ax.set_ylim(ymin - margin, ymax + margin)

    def style_map_axis(ax):
        ax.set_axis_off()
        ax.set_facecolor(PALETTE["background"])

    def load_penetration(year=year):
        path = PENETRATION_PATHS[str(year)]
        part = pd.read_csv(path, header=None, names=["log_eta"])
        part["year"] = str(year)
        part["class"] = np.rint(part["log_eta"]).astype(int)
        part = part.loc[part["class"].isin(PENETRATION_LABELS)]
        return part

    penetration = load_penetration(year)

    print(
        "OK: validation inputs ready "
        f"({len(penetration):,} penetration observations; "
        f"road network: {CARTO_PATH})"
    )

except Exception as exc:
    print(f"ERROR validation inputs: {type(exc).__name__}: {exc}")
    raise


OK: validation inputs ready (838 penetration observations; road network: article/data/uniq/carto.geojson)


In [66]:
# 6a_viasat_coverage.png - geolocated Viasat coverage on the full road cartography.
try:
    roads = read_roads()
    cid_col = road_id_column(roads)
    if year == 2020: via = pd.read_csv(VIASAT_FLUX_PATH, sep=";")
    if year == 2021: via = pd.read_csv(VIASAT_FLUX_PATH_2021, sep=";")
    if "cid" not in via.columns or "total_fluxes" not in via.columns:
        raise KeyError("Expected columns 'cid' and 'total_fluxes' in Viasat flux file")

    covered_ids = (
        via.groupby("cid", as_index=False)["total_fluxes"]
        .sum()
        .query("total_fluxes > 0")["cid"]
    )

    roads["is_viasat_covered"] = roads[cid_col].isin(covered_ids)
    roads_web = roads.to_crs(epsg=3857)

    full_network = roads_web
    covered_network = roads_web.loc[roads_web["is_viasat_covered"]]

    if covered_network.empty:
        raise ValueError("No Viasat-covered road segment found after merging cid/poly_cid")

    fig, ax = plt.subplots(figsize=(10.0, 8.0), constrained_layout=True)

    full_network.plot(
        ax=ax,
        color=PALETTE["roads_minor"],
        linewidth=1,
        alpha=1,
        zorder=2,
    )

    covered_network.plot(
        ax=ax,
        color=PALETTE["roads_main"],
        linewidth=1.5,
        alpha=1,
        zorder=3,
    )

    set_map_extent(ax, roads_web)
    add_positron_basemap(ax, roads_web, zoom=8)

    legend_handles = [
        Line2D([0], [0], color=PALETTE["roads_minor"], lw=3, label="Complete road cartography"),
        Line2D([0], [0], color=PALETTE["roads_main"], lw=3, label="Viasat-covered roads"),
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower left",
        fontsize=18,
        frameon=True,
        facecolor="white",
        edgecolor="#DDDDDD",
    )

    style_map_axis(ax)
    save_fig_local(fig, f"6a_viasat_coverage_{year}")

    print(
        f"OK: Viasat coverage plotted "
        f"({covered_network.shape[0]:,}/{full_network.shape[0]:,} road features covered)"
    )

except Exception as exc:
    print(f"ERROR 6a_viasat_coverage: {type(exc).__name__}: {exc}")
    raise


OK: saved 6a_viasat_coverage_2020.png and 6a_viasat_coverage_2020.pdf in article/tex/images
OK: Viasat coverage plotted (930/4,186 road features covered)


In [67]:
# 6b_penetration_statistics.png - statistical penetration as a palette-consistent pie chart.
try:
    order = [-1, -2, -3]

    counts = (
        penetration
        .groupby("class")
        .size()
        .reindex(order)
        .fillna(0)
        .astype(int)
    )

    labels = [PENETRATION_LABELS[k] for k in order]
    colors = [PENETRATION_COLORS[k] for k in order]

    fig, ax = plt.subplots(figsize=(12.2, 9.2), constrained_layout=True)

    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=labels,
        colors=colors,
        autopct=lambda pct: f"{pct:.1f}%" if pct > 0 else "",
        startangle=120,
        counterclock=False,
        labeldistance=1.08,
        pctdistance=0.64,
        wedgeprops={"linewidth": 1.5, "edgecolor": "white"},
        textprops={"fontsize": 25, "fontweight": "bold", "color": PALETTE["text"]},
    )

    for text in texts + autotexts:
        text.set_fontweight("bold")
        text.set_color(PALETTE["text"])

    save_fig_local(fig, f"6b_penetration_statistics_{year}")
    print("OK: penetration pie counts:", counts.to_dict())

except Exception as exc:
    print(f"ERROR 6b_penetration_statistics: {type(exc).__name__}: {exc}")
    raise


OK: saved 6b_penetration_statistics_2020.png and 6b_penetration_statistics_2020.pdf in article/tex/images
OK: penetration pie counts: {-1: 480, -2: 305, -3: 53}


In [68]:
# Helper plus 6c_cluster_2020.png - Viasat penetration clusters on the same cartography.
try:

    def plot_cluster_map(year, stem):

        # ------------------------------------------------------------------
        # Roads
        # ------------------------------------------------------------------
        roads = read_roads().to_crs(epsg=3857)

        # ------------------------------------------------------------------
        # Cluster data
        # ------------------------------------------------------------------
        clusters = gpd.read_file(CLUSTER_PATHS[year])

        if clusters.crs is None:
            clusters = clusters.set_crs(epsg=4326)

        clusters = clusters.to_crs(epsg=3857)

        if "penetration" not in clusters.columns:
            raise KeyError(
                f"'penetration' column not found in {CLUSTER_PATHS[year]}"
            )

        # ------------------------------------------------------------------
        # Keep only valid classes for plotting/extent
        # ------------------------------------------------------------------
        cluster_extent = clusters.loc[
            clusters["penetration"].isin([-1, -2, -3])
        ].copy()

        if cluster_extent.empty:
            cluster_extent = clusters.copy()

        # ------------------------------------------------------------------
        # Figure
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(
            figsize=(10.0, 8.0),
            constrained_layout=True,
        )

        # ------------------------------------------------------------------
        # Background roads
        # ------------------------------------------------------------------
        roads.plot(
            ax=ax,
            color=PALETTE["roads_minor"],
            linewidth=0.18,
            alpha=0.35,
            zorder=2,
        )

        # ------------------------------------------------------------------
        # Cluster overlays
        # ------------------------------------------------------------------
        for klass in [-3, -2, -1]:

            selected = clusters.loc[
                clusters["penetration"] == klass
            ]

            if selected.empty:
                continue

            selected.plot(
                ax=ax,
                color=CLUSTER_COLORS[klass],
                linewidth=2.2 if klass == -1 else 1.7,
                alpha=0.95,
                zorder=3 + abs(klass),
            )

        # ------------------------------------------------------------------
        # Automatic zoom based on cluster geometry
        # ------------------------------------------------------------------
        xmin, ymin, xmax, ymax = cluster_extent.total_bounds

        dx = xmax - xmin
        dy = ymax - ymin

        pad_x = dx * 0.08
        pad_y = dy * 0.08

        ax.set_xlim(xmin - pad_x, xmax + pad_x)
        ax.set_ylim(ymin - pad_y, ymax + pad_y)

        # ------------------------------------------------------------------
        # Basemap
        # ------------------------------------------------------------------
        add_positron_basemap(
            ax,
            cluster_extent,
            zoom=9,
        )

        # ------------------------------------------------------------------
        # Legend
        # ------------------------------------------------------------------
        handles = [
            Line2D(
                [0],
                [0],
                color=CLUSTER_COLORS[k],
                lw=4,
                label=PENETRATION_LABELS[k],
            )
            for k in [-1, -2, -3]
        ]

        ax.legend(
            handles=handles,
            loc="lower right",
            fontsize=15,
            frameon=True,
            facecolor="white",
            edgecolor="#DDDDDD",
        )

        # ------------------------------------------------------------------
        # Style + save
        # ------------------------------------------------------------------
        style_map_axis(ax)

        save_fig_local(fig, stem)

        print(
            f"OK: {year} clusters plotted; "
            f"{clusters['penetration'].value_counts().sort_index().to_dict()}"
        )

    # ----------------------------------------------------------------------
    # Run
    # ----------------------------------------------------------------------
    plot_cluster_map("2020", "6c_cluster_2020")
    plot_cluster_map("2021", "6d_cluster_2021")
    

except Exception as exc:
    print(f"ERROR 6c_cluster_2020: {type(exc).__name__}: {exc}")
    raise

OK: saved 6c_cluster_2020.png and 6c_cluster_2020.pdf in article/tex/images
OK: 2020 clusters plotted; {-4: 3, -3: 53, -2: 305, -1: 480, 0: 7}
OK: saved 6d_cluster_2021.png and 6d_cluster_2021.pdf in article/tex/images
OK: 2021 clusters plotted; {-4: 4, -3: 49, -2: 381, -1: 411}
